# PCAP Template Generation

This notebook demonstrates the PCAP traffic pattern templates for generating synthetic network traffic data using Rockfish's Entity Data Generator.

## Template Categories

### Normal Traffic (6 templates)
| Template | Description |
|----------|-------------|
| `web_browsing` | Standard HTTP/HTTPS sessions |
| `api_calls` | REST API request/response |
| `ssh_session` | Interactive SSH traffic |
| `file_transfer` | FTP/SMB file downloads |
| `database_query` | MySQL/PostgreSQL queries |
| `email` | SMTP/IMAP traffic |

### Anomalous Traffic (6 templates)
| Template | Description |
|----------|-------------|
| `port_scan` | Sequential port probing |
| `syn_flood` | SYN flood attack |
| `large_upload` | Data exfiltration |
| `beaconing` | C2 beaconing pattern |
| `dns_tunnel` | DNS tunneling attempt |
| `slow_loris` | Slow HTTP attack |

### Suspicious Traffic (5 templates)
| Template | Description |
|----------|-------------|
| `brute_force` | Auth brute force |
| `lateral_movement` | Internal reconnaissance |
| `data_staging` | Pre-exfil staging |
| `encrypted_tunnel` | TLS on non-standard port |
| `protocol_anomaly` | Protocol mismatch |

## Setup and Imports

In [ ]:
import sys
sys.path.insert(0, '..')

import rockfish as rf
import rockfish.actions as ra
from dotenv import load_dotenv
import pandas as pd
import numpy as np

# Import our templates
from templates import (
    TEMPLATE_REGISTRY,
    list_templates,
    get_template,
    create_normal_web_browsing_schema,
    create_anomaly_port_scan_schema,
    create_suspicious_brute_force_schema,
)

In [ ]:
# Connect to the Rockfish platform
load_dotenv()
conn = rf.Connection.from_env()

## List Available Templates

In [ ]:
# List all available templates
templates = list_templates()
print("Available PCAP Templates:")
print("=" * 50)
for category, template_names in templates.items():
    print(f"\n{category.upper()}:")
    for name in template_names:
        print(f"  - {name}")

## Normal Traffic: Web Browsing Example

This template generates standard HTTP/HTTPS web browsing sessions with:
- Full TCP handshake (SYN -> SYN-ACK -> ACK)
- 5-15 data packets (request + response)
- Graceful close (FIN -> FIN-ACK -> ACK)

In [ ]:
# Create schema for web browsing traffic
web_schema = create_normal_web_browsing_schema(
    n_sessions=50,
    n_client_hosts=20,
    n_server_hosts=10,
)

print(f"Schema created with {len(web_schema.entities)} entities:")
for entity in web_schema.entities:
    print(f"  - {entity.name}: {entity.cardinality} rows")

In [ ]:
# Generate web browsing data
config = ra.GenerateFromDataSchema.Config(
    schema=web_schema,
    upload_datasets=True,
)
generate = ra.GenerateFromDataSchema(config)

builder = rf.WorkflowBuilder()
builder.add(generate)
workflow = await builder.start(conn)
print(f"Workflow ID: {workflow.id()}")

In [ ]:
# Wait for completion and view logs
async for log in workflow.logs(level=rf.events.LogLevel.DEBUG):
    print(log)

In [ ]:
# Retrieve generated datasets
datasets = await workflow.datasets().collect()
print(f"Generated {len(datasets)} datasets")

web_session_df = None
for remote_ds in datasets:
    ds = await remote_ds.to_local(conn)
    if ds.name() == "tcp_session":
        web_session_df = ds.to_pandas()
        print(f"TCP Sessions: {len(web_session_df)} packets")

In [ ]:
# Analyze web browsing traffic
if web_session_df is not None:
    print("Web Browsing Traffic Analysis")
    print("=" * 50)
    print(f"\nTotal packets: {len(web_session_df)}")
    print(f"Unique sessions: {web_session_df['session_id'].nunique()}")
    print(f"Avg packets/session: {len(web_session_df) / web_session_df['session_id'].nunique():.1f}")
    
    print("\nTCP State Distribution:")
    print(web_session_df['tcp_state'].value_counts())
    
    print("\nPacket Type Distribution:")
    print(web_session_df['packet_type'].value_counts())

## Anomalous Traffic: Port Scan Example

This template generates port scanning attack patterns with:
- SYN -> RST (repeated across many ports)
- Very short sessions (1-2 packets)
- Many destination ports from same source
- No data transfer

In [ ]:
# Create schema for port scan traffic
scan_schema = create_anomaly_port_scan_schema(
    n_sessions=100,
    n_client_hosts=3,  # Few attackers
    n_server_hosts=20,  # Many targets
)

print(f"Schema created with {len(scan_schema.entities)} entities:")
for entity in scan_schema.entities:
    print(f"  - {entity.name}: {entity.cardinality} rows")

In [ ]:
# Generate port scan data
config = ra.GenerateFromDataSchema.Config(
    schema=scan_schema,
    upload_datasets=True,
)
generate = ra.GenerateFromDataSchema(config)

builder = rf.WorkflowBuilder()
builder.add(generate)
workflow = await builder.start(conn)
print(f"Workflow ID: {workflow.id()}")

In [ ]:
# Wait for completion
async for log in workflow.logs(level=rf.events.LogLevel.DEBUG):
    print(log)

In [ ]:
# Retrieve port scan data
datasets = await workflow.datasets().collect()

scan_session_df = None
for remote_ds in datasets:
    ds = await remote_ds.to_local(conn)
    if ds.name() == "tcp_session":
        scan_session_df = ds.to_pandas()
        print(f"Port Scan Sessions: {len(scan_session_df)} packets")

In [ ]:
# Analyze port scan traffic
if scan_session_df is not None:
    print("Port Scan Traffic Analysis")
    print("=" * 50)
    print(f"\nTotal packets: {len(scan_session_df)}")
    print(f"Unique sessions: {scan_session_df['session_id'].nunique()}")
    print(f"Avg packets/session: {len(scan_session_df) / scan_session_df['session_id'].nunique():.1f}")
    
    print("\nTCP State Distribution (should show RESET):")
    print(scan_session_df['tcp_state'].value_counts())
    
    print("\nPacket Type Distribution (should be mostly SYN/RST):")
    print(scan_session_df['packet_type'].value_counts())
    
    if 'target_port' in scan_session_df.columns:
        print(f"\nUnique target ports scanned: {scan_session_df['target_port'].nunique()}")

## Suspicious Traffic: Brute Force Example

This template generates authentication brute force patterns with:
- Multiple short sessions
- Same destination, different source ports
- SSH/RDP targets
- Quick failures (RST after auth attempt)

In [ ]:
# Create schema for brute force traffic
brute_schema = create_suspicious_brute_force_schema(
    n_sessions=100,
    n_client_hosts=2,  # Few attackers
    n_server_hosts=3,  # Few targets
)

print(f"Schema created with {len(brute_schema.entities)} entities:")
for entity in brute_schema.entities:
    print(f"  - {entity.name}: {entity.cardinality} rows")

In [ ]:
# Generate brute force data
config = ra.GenerateFromDataSchema.Config(
    schema=brute_schema,
    upload_datasets=True,
)
generate = ra.GenerateFromDataSchema(config)

builder = rf.WorkflowBuilder()
builder.add(generate)
workflow = await builder.start(conn)
print(f"Workflow ID: {workflow.id()}")

In [ ]:
# Wait for completion
async for log in workflow.logs(level=rf.events.LogLevel.DEBUG):
    print(log)

In [ ]:
# Retrieve brute force data
datasets = await workflow.datasets().collect()

brute_session_df = None
for remote_ds in datasets:
    ds = await remote_ds.to_local(conn)
    if ds.name() == "tcp_session":
        brute_session_df = ds.to_pandas()
        print(f"Brute Force Sessions: {len(brute_session_df)} packets")

In [ ]:
# Analyze brute force traffic
if brute_session_df is not None:
    print("Brute Force Traffic Analysis")
    print("=" * 50)
    print(f"\nTotal packets: {len(brute_session_df)}")
    print(f"Unique sessions: {brute_session_df['session_id'].nunique()}")
    print(f"Avg packets/session: {len(brute_session_df) / brute_session_df['session_id'].nunique():.1f}")
    
    print("\nTCP State Distribution (should show RESET and TIME_WAIT):")
    print(brute_session_df['tcp_state'].value_counts())
    
    print("\nPacket Type Distribution:")
    print(brute_session_df['packet_type'].value_counts())

## Using the Template Registry

You can dynamically access any template using the registry:

In [ ]:
# Get a template dynamically
ssh_template_fn = get_template("normal", "ssh_session")
ssh_schema = ssh_template_fn(n_sessions=30)

print(f"SSH Schema created with {len(ssh_schema.entities)} entities")

# Or iterate through all templates
print("\nAll templates in registry:")
for category, templates in TEMPLATE_REGISTRY.items():
    for template_name, template_fn in templates.items():
        print(f"  {category}/{template_name}")

## Traffic Pattern Comparison

Let's compare the characteristics of different traffic types:

In [ ]:
# Collect statistics if we have all three datasets
comparison_data = []

if web_session_df is not None:
    comparison_data.append({
        'Traffic Type': 'Normal (Web)',
        'Total Packets': len(web_session_df),
        'Unique Sessions': web_session_df['session_id'].nunique(),
        'Avg Packets/Session': round(len(web_session_df) / web_session_df['session_id'].nunique(), 1),
        'Dominant State': web_session_df['tcp_state'].value_counts().idxmax(),
    })

if scan_session_df is not None:
    comparison_data.append({
        'Traffic Type': 'Anomalous (Port Scan)',
        'Total Packets': len(scan_session_df),
        'Unique Sessions': scan_session_df['session_id'].nunique(),
        'Avg Packets/Session': round(len(scan_session_df) / scan_session_df['session_id'].nunique(), 1),
        'Dominant State': scan_session_df['tcp_state'].value_counts().idxmax(),
    })

if brute_session_df is not None:
    comparison_data.append({
        'Traffic Type': 'Suspicious (Brute Force)',
        'Total Packets': len(brute_session_df),
        'Unique Sessions': brute_session_df['session_id'].nunique(),
        'Avg Packets/Session': round(len(brute_session_df) / brute_session_df['session_id'].nunique(), 1),
        'Dominant State': brute_session_df['tcp_state'].value_counts().idxmax(),
    })

if comparison_data:
    comparison_df = pd.DataFrame(comparison_data)
    print("Traffic Pattern Comparison")
    print("=" * 80)
    print(comparison_df.to_string(index=False))

## Summary

This notebook demonstrated the PCAP template system with 17 different traffic patterns:

### Key Features:

1. **Stateful TCP Sessions** - Each template uses a state machine to model realistic TCP behavior
2. **Configurable Parameters** - Number of sessions, hosts, and time ranges can be customized
3. **Traffic Categories** - Normal, Anomalous, and Suspicious patterns for security analysis
4. **Ground Truth Labels** - Each session can be labeled by its template type

### Next Steps:

- Use `02_generate_blended_pcap.py` to create week-long datasets with mixed traffic
- Use `03_create_baseline_pcap.py` to create a baseline without Rockfish
- Build a PCAP analysis agent to detect these patterns